# 🚢 Notebook 3: Bulkhead Isolation

Ship hulls are divided into watertight **bulkheads** so one flooded compartment doesn't sink the whole vessel. In services, we split the worker pool the same way: one slow dependency can't starve every caller of every other dependency.

**The failure mode to prevent:** one misbehaving downstream holds every thread/connection in a shared pool. Callers to *unrelated* downstreams queue behind them and time out too — a local outage becomes a global one.

## 🛠️ Setup

```bash
cd 04-patterns/resilience
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 🟥 BAD: a single shared thread pool

Four slow callers fill the shared pool. A fast call arrives — it should take 50 ms but has to wait for a slow one to finish first.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def slow_dep(): time.sleep(2.0); return 'slow'
def fast_dep(): time.sleep(0.05); return 'fast'

shared = ThreadPoolExecutor(max_workers=4)

t0 = time.perf_counter()
# 4 slow callers fill the pool
for _ in range(4):
    shared.submit(slow_dep)
# Now a fast caller arrives — but the pool is full
fut = shared.submit(fast_dep)
print('fast result:', fut.result(), 'after', round(time.perf_counter()-t0, 2), 's')
shared.shutdown(wait=True)


Even though the fast call itself takes 50 ms, it waited ~2 s for a slow slot — a *latency injection* across unrelated workloads.

## 🟩 GOOD: per-dependency pools

Give each downstream its own pool. The slow one can saturate its own bulkhead; the fast pool is untouched.

In [ ]:
slow_pool = ThreadPoolExecutor(max_workers=2)
fast_pool = ThreadPoolExecutor(max_workers=2)

t0 = time.perf_counter()
for _ in range(4):
    slow_pool.submit(slow_dep)   # only slow_pool fills up
fut = fast_pool.submit(fast_dep) # fast_pool is empty — runs immediately
print('fast result:', fut.result(), 'after', round(time.perf_counter()-t0, 2), 's')
slow_pool.shutdown(wait=False); fast_pool.shutdown(wait=False)


Slow calls stay isolated; the fast endpoint is unaffected.

## 🧵 Async equivalent: `asyncio.Semaphore`

In async Python we don't use thread pools — we use **semaphores** to cap concurrency per dependency. Same idea: one slow downstream can only hold N slots at a time.

In [ ]:
import asyncio

async def call_with_bulkhead(name, sem, duration):
    async with sem:                    # wait for a slot in THIS bulkhead
        await asyncio.sleep(duration)
        return name

async def demo():
    slow_sem = asyncio.Semaphore(2)    # at most 2 concurrent calls to slow dep
    fast_sem = asyncio.Semaphore(2)    # at most 2 concurrent calls to fast dep

    t0 = time.perf_counter()
    tasks = [
        # 4 slow calls compete for the 2-slot slow_sem
        *[call_with_bulkhead('slow', slow_sem, 2.0) for _ in range(4)],
        # 1 fast call gets its own bulkhead — runs immediately
        call_with_bulkhead('fast', fast_sem, 0.05),
    ]
    results = await asyncio.gather(*tasks)
    print('done in', round(time.perf_counter()-t0, 2), 's, results:', results)

await demo()


## 🧠 Where to apply

- **One pool per downstream** — isolate the payment service from the recommendation service.
- **One pool per tenant** — a noisy customer can't starve others (multi-tenant SaaS).
- **One pool per endpoint criticality** — health-check endpoints never wait behind report-generation endpoints.
- **Always combine with timeouts** — a bulkhead limits *how many* stuck tasks you have; timeouts limit *how long* each is stuck (notebook 4).
- **Combine with circuit breakers** — each bulkhead gets its own breaker. When the payment bulkhead trips, recommendations keep working.

## ⚖️ What it costs, and when not to bother

Partitioning is not free — you are deliberately giving up the efficiency of a shared pool:

- **Lower utilization.** Split 8 threads into two pools of 4, and a burst on one side queues while 4 threads sit idle on the other. That is the point, and it is also the price. Total capacity has to grow to keep the same peak throughput.
- **More knobs to get wrong.** Each pool is a number someone has to size, and the sum has to stay inside your real limits (a pool per dependency is useless if they all share one 10-connection database pool).
- **`ThreadPoolExecutor` is not a bulkhead by itself.** Its queue is **unbounded**, so at saturation callers don't fail fast — they queue forever, which is the unbounded-queue failure from the rate-limiting lab. A bulkhead you can actually rely on **rejects** when full; see notebook 5 for the `Semaphore(n)` + `acquire(blocking=False)` version.

**When NOT to bulkhead:** a service with a single downstream (there's nothing to isolate *from*), or when the partitions would be so small that normal traffic starts hitting them. If each pool is 2 threads, you haven't built a bulkhead, you've built a bottleneck.

> Bulkheads are about *blast radius*. They don't make failures less likely — they make each failure **local** instead of **global**.